Chains allow us to combine multiple components together to create a single, coherent application


In [ ]:
!pip install -q  langchain-openai==0.1.8 langchain_community==0.2.4 tiktoken==0.7.0 langchain==0.2.3


In [ ]:
from google.colab import userdata
openai_api_key = userdata.get('openai_api_key')


In [ ]:
from langchain.chains import LLMChain
# from langchain.llms import OpenAI
# from langchain.chat_models import ChatOpenAI
from langchain_openai import ChatOpenAI

from langchain.prompts import PromptTemplate
prompt = PromptTemplate.from_template("What is the best name for a company that makes {product}?")
llm = ChatOpenAI(openai_api_key=openai_api_key,model='gpt-4-1106-preview' ,temperature=0)
chain = LLMChain(llm=llm, prompt=prompt)

In [ ]:
llm.invoke(prompt.format(product="colorful socks"))

In [ ]:
chain.invoke({'product':"colorful socks"})

In [ ]:
chain.invoke("colorful socks", return_only_outputs=True)

In [ ]:
from langchain.chains import LLMMathChain

llm_math = LLMMathChain.from_llm(llm, verbose=True)
#


In [ ]:
llm_math.prompt.template

In [ ]:
i = 0
math_problems = ["What is 13 raised to the .3432 power?",
                 " Determine the result of raising 2 to the power of 10 and then subtracting the square root of 81."]
# llm_math.invoke()
llm_math.invoke(math_problems[i])

In [ ]:
from langchain.schema import HumanMessage
llm.invoke([HumanMessage(content=math_problems[i])])

[see more legacy chain](https://python.langchain.com/docs/modules/chains/#legacy-chains)

LangChain Expression Language (**LCEL**)

In [ ]:
from langchain.prompts import PromptTemplate
template = """
 Please translate the following word from {input_language} to {output_language}
 word: {word}
 Answer: ?"""
prompt = PromptTemplate(template=template, input_variables=["input_language", 'output_language', 'word'])


In [ ]:
translator_chain = prompt | llm
translator_chain.invoke({'input_language':'English', 'output_language':'Farsi', 'word':"Hypothesis"})

In [ ]:
response = translator_chain.invoke({'input_language':'English', 'output_language':'Farsi', 'word':"Hypothesis"}).content
response

In [ ]:
from langchain_core.output_parsers import StrOutputParser
prompt = PromptTemplate(template='answer {question} honestly', input_variables=['question'])
chain = prompt | llm | StrOutputParser()
chain.invoke({'question':'who was Amirkabir?'})

In [ ]:
from langchain_core.output_parsers import JsonOutputParser

parser = JsonOutputParser()
prompt = PromptTemplate(template='answer {question} honestly. \n{format_instructions}\n',
                        input_variables=['question'], partial_variables={"format_instructions": parser.get_format_instructions()})

chain = prompt | llm | parser
chain.invoke({'question':'who was Amirkabir?'})

In [ ]:
inputs = [
    {'input_language':'English', 'output_language':'Farsi', 'word':"Hypothesis"},
    {'input_language':'Farsi', 'output_language':'Arabic', 'word':"درخت"}
         ]
response = translator_chain.batch(inputs)
response

In [ ]:
async for chunk in chain.astream({'question':'who was Amir kabir?'},):
  print(chunk, end='\n')


In [ ]:
chunks = []
for chunk in chain.stream({'question':'who was Amirkabir?'}):
  chunks.append(chunk)
  print(chunk, end="\n", flush=True)

In [ ]:

model_parser = llm | StrOutputParser()

### terapist chain
terapist_prompt = PromptTemplate(template="You are a helpful therapy assistant. Please answer {question}.", input_variables=["question"])
terapist = terapist_prompt | model_parser


### code developer chain
code_developer_prompt = PromptTemplate(template='you are an excellent {language} developer. please answer {question}', input_variables=['language', 'question'])
code_developer = code_developer_prompt | model_parser



### general chain
general_chain = (
    PromptTemplate.from_template(
        """Respond to the following question:Question: {question} Answer:"""
    )
    | model_parser
)


In [ ]:
from langchain_core.runnables import RunnableBranch, RunnableParallel
### A RunnableBranch is initialized with a list of (condition, runnable) pairs and a default runnable.

branch = RunnableBranch(
    (lambda x: "terapy" ==x["topic"].lower(), terapist),
    (lambda x: "code" in x["topic"].lower(), code_developer),
    general_chain
)

In [ ]:
full_chain = {"topic": lambda x: x["topic"],
              "question": lambda x: x["question"], "language":lambda x: x.get('language', None)} | branch



In [ ]:
full_chain.invoke({'topic': 'code', 'question': 'What is the difference between a list and a tuple?', 'language': 'python'})

In [ ]:

template = """Just classify the topic of the following question: {question} into one of these three categories:
 ['code', 'therapy', 'general'] with just one category without any explanation."""

prompt = PromptTemplate(template=template,
                        input_variables=['question'])
classifier_chain = prompt | model_parser



In [ ]:
classifier_chain.invoke({'question': 'I lost my father 4 years ago and I still miss him'})

In [ ]:
from operator import itemgetter
final_chain = ({'topic': classifier_chain, 'question': itemgetter("question"), 'language': lambda x: x.get("language",None)} | branch)


In [ ]:
final_chain.invoke({'question': 'I lost my father 4 years ago and I still miss him'})

In [ ]:
final_chain.invoke({ 'question': 'What is the difference between a list and a tuple?', 'language': 'python'})